In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from crewai import LLM

llm = LLM(
    model="gpt-4o",
    temperature=0.7,
    max_tokens=150,
)

In [3]:
from crewai.tools import BaseTool # BaseTool - the parent class for all CrewAI custom tools.
# CrewAI lets you build tools that your agents can call.
# Every custom tool MUST inherit from BaseTool.

class ReplaceJargonsTool(BaseTool):
    name: str = "Jargon replacement tool"
    description : str = "Replaces jargon with more specific terms. "

    def _run(self, email: str) -> str:
        # A dictionary of corporate jargons → their expanded, clearer meanings.
        replacements = {
            "PRX": "Project Phoenix (internal AI revamp project)",
            "TAS": "technical architecture stack",
            "DBX": "client database cluster",
            "SDS": "Smart Data Syncer",
            "SYNCBOT": "internal standup assistant bot",
            "WIP": "in progress",
            "POC": "proof of concept",
            "ping": "reach out"
        }
        suggestions = [] # Create an empty list to collect suggestions for the user. Every time jargon is found, an explanation is appended here.
        email_lower = email.lower()
        for jargon, replacement in replacements.items():
            if jargon.lower() in email_lower:
                suggestions.append(f"Consider replacing '{jargon}' with '{replacement}'")

        return "\n".join(suggestions) if suggestions else "No jargon or internal abbreviations detected."

# Create an instance of your tool. Agents cannot use a tool class — They can use only an instance.
jt = ReplaceJargonsTool()

original_email = """
looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.
Let's sync up tomorrow if SYNCBOT allows 😄. ping me if any blockers.
"""

jt.run(original_email) # Run the tool with your email text.

Using Tool: Jargon replacement tool


"Consider replacing 'PRX' with 'Project Phoenix (internal AI revamp project)'\nConsider replacing 'TAS' with 'technical architecture stack'\nConsider replacing 'SDS' with 'Smart Data Syncer'\nConsider replacing 'SYNCBOT' with 'internal standup assistant bot'\nConsider replacing 'ping' with 'reach out'"

In [4]:
from crewai import Agent, Task, Crew

email_assistant = Agent(
    role="Email Assistant Agent",
    goal="Improve emails and make them sound professional and clear",
    backstory="A highly experienced communication expert skilled in professional email writing",
    verbose=True,
    tools=[jt],
    llm=llm
)

email_task = Task(
    description=f"""Take the following rough email and rewrite it into a professional and polished version.
    Expand abbreviations:
    '''{original_email}'''""",
    agent=email_assistant,
    expected_output="A professional written email with proper formatting and content.",
)

crew = Crew(
    agents=[email_assistant],
    tasks=[email_task],
    verbose=True
)

result = crew.kickoff()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bce8f739-55c7-4434-a793-d099e39c8043                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the following rough email and rewrite it into a professional and polished version.                  │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.                      │
│  Let's sync up tomorrow if SYNCBOT allows 😄. ping me if any blockers.                                          │
│  '''                                                                                                            │
│  ID: 5b33ee9c-8efb-47e4-981b-96eb266df968                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Email Assistant Agent                                                                                   │
│                                                                                                                 │
│  Task: Take the following rough email and rewrite it into a professional and polished version.                  │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.                      │
│  Let's sync up tomorrow if SYNCBOT allows 😄. ping me if any blockers.                                          │
│  '''                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: Jargon replacement tool                                                                                  │
│  Args: {"email": "looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.     │
│  Let's sync up tomorrow if SYNCBOT allows \ud83d\ude04. ping me if any blockers."}                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Consider replacing 'PRX' with 'Project Phoenix (internal AI revamp project)'                                   │
│  Consider replacing 'TAS' with 'technical architecture stack'                                                   │
│  Consider replacing 'SDS' with 'Smart Data Syncer'                                                              │
│  Consider replacing 'SYNCBOT' with 'internal standup assistant bot'                                             │
│  Consider replacing 'ping' with 'reach out'                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Email Assistant Agent                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Project Updates and Coordination                                                                      │
│                                                                                                                 │
│  Hi Priya,                                                                                                      │
│                                                                                                                 │
│  I am including you in this conversation. The updates for the technical architecture stack and Project Phoenix  │
│  (internal AI revamp project) are available in the presentation deck. The estimated time of arrival for the     │
│  Smart Data Syncer integration is this Friday.                                                                  │
│                                                                                                                 │
│  Let's plan to synchronize tomorrow if the internal standup assistant bot allows. Please reach out to me if     │
│  you encounter any blockers.                                                                                    │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Take the following rough email and rewrite it into a professional and polished version.                        │
│      Expand abbreviations:                                                                                      │
│      '''                                                                                                        │
│  looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.                      │
│  Let's sync up tomorrow if SYNCBOT allows 😄. ping me if any blockers.                                          │
│  '''                                                                                                            │
│  Agent:                                                                                                         │
│  Email Assistant Agent                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Subject: Project Updates and Coordination

Hi Priya,

I am including you in this conversation. The updates for the technical architecture stack and Project Phoenix (internal AI revamp project) are available in the presentation deck. The estimated time of arrival for the Smart Data Syncer integration is this Friday.

Let's plan to synchronize tomorrow if the internal standup assistant bot allows. Please reach out to me if you encounter any blockers.

Best regards,

[Your Name]


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bce8f739-55c7-4434-a793-d099e39c8043                                                                           │
│  Final Output: Subject: Project Updates and Coordination                                                        │
│                                                                                                                 │
│  Hi Priya,                                                                                                      │
│                                                                                                                 │
│  I am including you in this conversation. The updates for the technical architecture stack and Project Phoenix  │
│  (internal AI revamp project) are available in the presentation deck. The estimated time of arrival for the     │
│  Smart Data Syncer integration is this Friday.                                                                  │
│                                                                                                                 │
│  Let's plan to synchronize tomorrow if the internal standup assistant bot allows. Please reach out to me if     │
│  you encounter any blockers.                                                                                    │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ cda1e451-2917-4393-ac8b-66e56de00341                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/cda1e451-2917-439 │
│ 3-ac8b-66e56de00341?access_code=TRACE-b2bc98241f                             │
│ 🔑 Access Code: TRACE-b2bc98241f                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
